# CS 3892 / 5892 — Session 9 · Inductive Invariants and SMV

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ttj/cs3892-examples/blob/main/notebooks/cs3892-2026-09-24-inductive-invariants-and-smv.ipynb)

**Thursday, September 24, 2026.** Tuesday ended at a bound. Today we get past it.

**The one move:** stop asking *"is there a bad run of length ≤ k?"* once per `k`,
and start asking two questions that mention **no `k` at all`**.

## Setup

Run this once.

In [ ]:
# --- Setup: find the repo (clone on Colab), install Z3, define helpers -------
import os, subprocess, sys, pathlib

REPO_URL = "https://github.com/ttj/cs3892-examples.git"
SESSION  = "cs3892-2026-09-24-inductive-invariants-and-smv"

def _find_repo():
    """Walk up from the CWD looking for the repo; otherwise clone it."""
    here = pathlib.Path.cwd()
    for p in [here, *here.parents]:
        if (p / "sessions" / SESSION).is_dir():
            return p
    dest = pathlib.Path("/content/cs3892-examples") if pathlib.Path("/content").is_dir() \
           else pathlib.Path.cwd() / "cs3892-examples"
    if not (dest / "sessions" / SESSION).is_dir():
        print(f"$ git clone {REPO_URL} {dest}")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
    return dest

ROOT = _find_repo()
os.chdir(ROOT)
print("repo:", ROOT)

try:
    import z3
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "z3-solver"], check=True)
    import z3
print("Z3", z3.get_version_string())

SM = ROOT / "sessions" / SESSION / "smt2"
PYD = ROOT / "sessions" / SESSION / "python"

def show(path):
    """Print a source file, so you can read what you are about to run."""
    print(f"--- {pathlib.Path(path).name} " + "-" * max(0, 60 - len(pathlib.Path(path).name)))
    print(pathlib.Path(path).read_text().rstrip())
    print()

def run(path, show_source=True):
    """Run one example and stream its output. Raises if it does not pass.

    .smt2 goes through scripts/run_smt2.py, which checks the file's own
    `; EXPECT:` contract. .py is executed directly and asserts internally.
    The pip wheel for Z3 ships no `z3` CLI, which is why .smt2 is run through
    the Python bindings rather than a shell command -- identical everywhere.
    """
    path = pathlib.Path(path)
    if show_source:
        show(path)
    cmd = ([sys.executable, "scripts/run_smt2.py", str(path)] if path.suffix == ".smt2"
           else [sys.executable, str(path)])
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout.rstrip())
    if r.stderr.strip():
        print(r.stderr.rstrip(), file=sys.stderr)
    if r.returncode != 0:
        raise RuntimeError(f"{path} failed")
    return r.stdout

print("ready — helpers: show(path), run(path)")

## 1. The BMC encoding, written out

Slides 6–8. The formula at each `k`, printed rather than described — then the
bound pushed out to 50.

Fifty `unsat` verdicts is fifty facts about runs of length ≤ 50. It is **not**
a proof, and no amount of pushing `k` makes it one.

In [ ]:
run(SM  / "01_bmc_k3.smt2")
run(PYD / "01_bmc_worked.py")

## 2. Two queries that settle every `k` at once

Slides 18–20. Three obligations, each a *validity* question, each turned into an
unsatisfiability question exactly as in session 7:

```
INITIATION    I(x)  =>  Inv(x)
CONSECUTION   Inv(x) ∧ T(x,x')  =>  Inv(x')
SUFFICIENCY   Inv(x)  =>  P(x)
```

Count the variables in `02_induction.smt2`: **two**, not `k+1`. The formula does
not grow with the length of the run, because it never mentions it.

In [ ]:
run(SM  / "02_induction.smt2")
run(PYD / "02_inductive_check.py")

## 3. Invariant, but not inductive

Slide 21, and the idea the whole lecture turns on.

```
I(x) = (x = 0)    T(x,x') = (x' = x + 2)    P(x) = (x ≠ 5)
```

`P` is a true invariant — every reachable `x` is even and 5 is odd — and bounded
checking agrees at every `k`. But consecution quantifies over **every** `x`
satisfying `P`, reachable or not, and `x = 3` steps straight onto 5.

That is a **counterexample to induction (CTI)**: a state that satisfies the
property, leaves it in one step, and is not reachable at all.

In [ ]:
run(SM  / "03_not_inductive.smt2")
run(PYD / "03_not_inductive.py")

## 4. Strengthening

Slide 22. The cure is counter-intuitive the first time: make the claim
**stronger**, not weaker.

`Inv(x) = "x is even"` is stronger than `x ≠ 5` — a *smaller* set of states — and
shrinking it until it fits inside the reachable set is exactly what makes
consecution go through. `Inv ⇒ P` keeps the thing you actually wanted.

Nothing about the solver changed. The stronger claim was the **easier** one to prove.

In [ ]:
run(PYD / "04_strengthen.py")

## 5. The same machine in SMV

Slides 24–25. `smv/counter.smv` is the counter written the way a model checker
wants it. `ASSIGN init/next` **is** the transition relation, with a syntax that
names the variable once instead of writing `x` and `x'` side by side.

Three `SPEC`s: `AG (x <= 10)` is the safety property, `EF (x = 10)` says the
machine really gets there, and `AG (x <= 9)` is **false on purpose** — ask for it
and you get a counterexample trace, the same object Z3 handed back as a model.

**Run it right here:** the next cell fetches NuSMV from FBK (free, LGPL — a few
seconds) and checks all three SPECs. **Or in your browser, nothing to install:**
<https://bit.ly/fmaiv_smvis> — paste the file in, or explore the model interactively.

| | |
|---|---|
| **HW1** | due tonight |
| **Quiz 3, Oct 6** | transition systems, invariants, SMV — this session |
| **HW2** | temporal logic and model checking in nuXmv, out Oct 1 |

Files: `github.com/ttj/cs3892-examples/tree/main/sessions/cs3892-2026-09-24-inductive-invariants-and-smv`


In [ ]:
# --- NuSMV: fetch the official FBK build if it is not already installed ------
# NuSMV is free (LGPL) from nusmv.fbk.eu. nuXmv is license-gated, so it is not
# installed here -- the smvis link above covers the browser route.
# 2.7.1 is tried first; it needs glibc >= 2.38 and libedit, which an older Colab
# image may lack, so the static 2.6.0 build is the fallback. Same verdicts.
import re, shutil, subprocess, os, pathlib

SMV = ROOT / "sessions" / SESSION / "smv"
BUILDS = [("2.7.1", "https://nusmv.fbk.eu/distrib/2.7.1/NuSMV-2.7.1-linux64.tar.xz", "J"),
          ("2.6.0", "https://nusmv.fbk.eu/distrib/NuSMV-2.6.0-linux64.tar.gz",       "z")]

def _works(exe):
    """True only if this binary actually model-checks the file -- a missing
    library makes it exit 127 with an error that still contains "NuSMV"."""
    try:
        r = subprocess.run([exe, str(SMV / "counter.smv")], capture_output=True, text=True, timeout=60)
        return "-- specification" in r.stdout
    except Exception:
        return False

NUSMV = shutil.which("NuSMV")
if not (NUSMV and _works(NUSMV)):
    NUSMV = None
    if os.geteuid() == 0 and shutil.which("apt-get"):      # Colab runs as root
        subprocess.run("apt-get install -y -qq libedit2 >/dev/null 2>&1", shell=True)
    for ver, url, z in BUILDS:
        dest = pathlib.Path.home() / ".local" / f"nusmv-{ver}"
        dest.mkdir(parents=True, exist_ok=True)
        subprocess.run(f"curl -sSL {url} | tar -x{z} -C {dest} --strip-components=1",
                       shell=True, check=True)
        if _works(str(dest / "bin" / "NuSMV")):
            NUSMV = str(dest / "bin" / "NuSMV")
            break
        print(f"NuSMV {ver} will not start on this machine -- trying the next build")
assert NUSMV, "could not install NuSMV"
print("NuSMV:", NUSMV)

show(SMV / "counter.smv")
out = subprocess.run([NUSMV, str(SMV / "counter.smv")], capture_output=True, text=True).stdout
print("\n".join(l for l in out.splitlines() if not l.startswith("***")))

verdicts = re.findall(r"-- specification (.*?)\s+is (true|false)", out)
assert verdicts == [("AG x <= 10", "true"), ("EF x = 10", "true"), ("AG x <= 9", "false")], verdicts
print("all three verdicts match slide 25")
